# SAM-Med2D Fine-tuned — Test Only (upload checkpoint)

Chạy tuần tự: **Setup → (%%writefile cells) → Chuẩn bị → Test A/B/C → Kết quả image-level**

In [ ]:
# ══════════════════════════════════════════════════════
# SETUP — Test Only (không fine-tune)
# ══════════════════════════════════════════════════════
%cd /kaggle/working
!git clone https://github.com/OpenGVLab/SAM-Med2D/
%cd /kaggle/working/SAM-Med2D
!pip install -e . -q
!pip install -q albumentations gdown

# Dataset
import os, gdown
os.makedirs('checkpoint', exist_ok=True)
!gdown "https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW" \
    -O /tmp/dataset_BTXRD.zip -q
!unzip -oq /tmp/dataset_BTXRD.zip -d /kaggle/working/SAM-Med2D/

# ── Checkpoint (điền vào trước khi chạy) ────────────────────────────────
# Upload SAM-Med2D FINE-TUNED checkpoint lên Google Drive rồi điền ID vào đây
CKPT_ID = ''      # ← điền ID Google Drive, ví dụ: '1abc...'
CHECKPOINT = '/kaggle/working/SAM-Med2D/checkpoint/best_sam.pth'
assert CKPT_ID, '❌ Chưa điền CKPT_ID — upload fine-tuned SAM checkpoint lên Drive rồi điền ID'
gdown.download(f'https://drive.google.com/uc?id={CKPT_ID}', CHECKPOINT, quiet=False)
assert os.path.exists(CHECKPOINT)
print(f'\n✅ Checkpoint: {CHECKPOINT}  ({os.path.getsize(CHECKPOINT)//1024} KB)')


## Phần 2 – Thí nghiệm Đánh giá

Ba kịch bản prompt bbox được kiểm tra:
- **A – Zoom-Out ~30%**: bbox mở rộng đều 4 phía 30% so với tight bbox GT (tương ứng Gaussian heatmap của PGA)
- **B – Shift ~30%**: bbox lệch vị trí ~30% (mô phỏng người dùng chú thích không chính xác)
- **C – Mixed 70/30**: 70% zoom-out + 30% shift (kịch bản thực tế)

### 2.0 – Chuẩn bị (chạy 1 lần trước khi test)

In [11]:
%cd /kaggle/working/SAM-Med2D
import os, json, glob

# ── 1. Tạo label2image_test.json ────────────────────────────────────────────
def create_evaluation_json(split="test"):
    mask_dir = f"dataset_BTXRD/{split}/masks"
    img_dir  = f"dataset_BTXRD/{split}/images"
    label2image = {}

    if not os.path.exists(mask_dir) or not os.path.exists(img_dir):
        print(f"[!] Không tìm thấy thư mục cho tập: '{split}'"); return

    img_dict = {}
    for f in os.listdir(img_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_dict[os.path.splitext(f)[0]] = f

    for mask_name in sorted(os.listdir(mask_dir)):
        if not mask_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue
        mask_base = os.path.splitext(mask_name)[0]
        img_base  = mask_base.split("_")[0] if "_" in mask_base else mask_base
        if img_base in img_dict:
            label2image[f"dataset_BTXRD/{split}/masks/{mask_name}"] =                         f"dataset_BTXRD/{split}/images/{img_dict[img_base]}"

    out = f"dataset_BTXRD/label2image_{split}.json"
    with open(out, "w", encoding="utf-8") as f_out:
        json.dump(label2image, f_out, indent=4, ensure_ascii=False)
    print(f"[*] label2image_{split}.json — {len(label2image)} mẫu → {out}")

create_evaluation_json("test")
create_evaluation_json("val")

# ── 2. Load CHECKPOINT (ưu tiên best > last, Drive > local) ──────────────────
def find_ckpt(folder):
    for name in ["best_sam.pth", "last_sam.pth"]:
        p = os.path.join(folder, name)
        if os.path.exists(p): return p
    return None

CHECKPOINT = (find_ckpt("/kaggle/working/drive/MyDrive/model") or
              find_ckpt("/kaggle/working/SAM-Med2D/workdir/models/sam-med2d") or "")
if CHECKPOINT:
    src = "Drive" if "MyDrive" in CHECKPOINT else "Local"
    print(f"[{src}] ✅ {CHECKPOINT}")
else:
    print("❌ Không tìm thấy checkpoint. Chạy training trước.")

/kaggle/working/SAM-Med2D
[*] label2image_test.json — 232 mẫu → dataset_BTXRD/label2image_test.json
[*] label2image_val.json — 239 mẫu → dataset_BTXRD/label2image_val.json
[Drive] ✅ /kaggle/working/drive/MyDrive/model/best_sam.pth


In [12]:
%%writefile /kaggle/working/SAM-Med2D/metrics.py
import torch
import torch.nn as nn


class SegMetrics(nn.Module):
    """
    SegMetrics(pred, label, metrics)
    pred, label : Tensor [B, 1, H, W]  (logits or 0/1)
    metrics     : list of str in {'iou','dice','precision','recall'}
    Usage       : bm = SegMetrics(pred, label, ['iou','dice','precision','recall'])
                  val = float(bm[i])
    """

    def __init__(self, pred, label, metrics):
        super().__init__()
        pred_bin  = (pred  > 0).float()
        label_bin = (label > 0).float()

        B = pred_bin.shape[0]
        p = pred_bin.view(B, -1)
        g = label_bin.view(B, -1)

        tp = (p * g).sum(dim=1)                 # [B]
        fp = (p * (1 - g)).sum(dim=1)
        fn = ((1 - p) * g).sum(dim=1)
        eps = 1e-7

        iou       = (tp / (tp + fp + fn + eps)).mean()
        dice      = (2 * tp / (2 * tp + fp + fn + eps)).mean()
        precision = (tp / (tp + fp + eps)).mean()
        recall    = (tp / (tp + fn + eps)).mean()

        self._map = {
            'iou':       iou,
            'dice':      dice,
            'precision': precision,
            'recall':    recall,
        }
        self._results = [self._map[m] for m in metrics]

    def __getitem__(self, idx):
        return self._results[idx]

    def __len__(self):
        return len(self._results)


Overwriting /kaggle/working/SAM-Med2D/metrics.py


In [13]:
%%writefile /kaggle/working/SAM-Med2D/test.py
from segment_anything import sam_model_registry
import torch.nn as nn
import torch
import argparse
import os
from utils import FocalDiceloss_IoULoss, generate_point, save_masks
from torch.utils.data import DataLoader
from DataLoader import TestingDataset
from metrics import SegMetrics
import time
from tqdm import tqdm
import numpy as np
from torch.nn import functional as F
import logging
import datetime
import cv2
import random
import csv
import json
from scipy.ndimage import binary_erosion, distance_transform_edt

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--work_dir", type=str, default="workdir", help="work dir")
    parser.add_argument("--run_name", type=str, default="sammed", help="run model name")
    parser.add_argument("--batch_size", type=int, default=1, help="batch size")
    parser.add_argument("--image_size", type=int, default=256, help="image_size")
    parser.add_argument('--device', type=str, default='cuda')
    parser.add_argument("--data_path", type=str, default="data_demo", help="train data path")
    parser.add_argument("--metrics", nargs='+', default=['iou', 'dice', 'precision', 'recall'], help="metrics")
    parser.add_argument("--model_type", type=str, default="vit_b", help="sam model_type")
    parser.add_argument("--sam_checkpoint", type=str, default="pretrain_model/sam-med2d_b.pth", help="sam checkpoint")
    parser.add_argument("--boxes_prompt", type=bool, default=True, help="use boxes prompt")
    parser.add_argument("--point_num", type=int, default=1, help="point num")
    parser.add_argument("--iter_point", type=int, default=1, help="iter num")
    parser.add_argument("--multimask", type=bool, default=True, help="ouput multimask")
    parser.add_argument("--encoder_adapter", type=bool, default=True, help="use adapter")
    parser.add_argument("--prompt_path", type=str, default=None, help="fix prompt path")
    parser.add_argument("--save_pred", type=bool, default=False, help="save result")
    parser.add_argument("--prompt_mode", type=str, default="zoom_out",
                        choices=['zoom_out', 'shift', 'mixed_7_3'],
                        help="Bbox prompt scenario")
    parser.add_argument("--zoom_ratio", type=float, nargs=2, default=[0.15, 0.45])
    parser.add_argument("--shift_ratio", type=float, default=0.30)
    args = parser.parse_args()
    if args.iter_point > 1:
        args.point_num = 1
    return args

def to_device(batch_input, device):
    device_input = {}
    for key, value in batch_input.items():
        if value is not None:
            if key=='image' or key=='label':
                device_input[key] = value.float().to(device)
            elif type(value) is list or type(value) is torch.Size:
                device_input[key] = value
            else:
                device_input[key] = value.to(device)
        else:
            device_input[key] = value
    return device_input

def postprocess_masks(low_res_masks, image_size, original_size):
    ori_h, ori_w = original_size
    masks = F.interpolate(low_res_masks, (image_size, image_size),
                          mode="bilinear", align_corners=False)
    if ori_h < image_size and ori_w < image_size:
        top  = torch.div((image_size - ori_h), 2, rounding_mode='trunc')
        left = torch.div((image_size - ori_w), 2, rounding_mode='trunc')
        masks = masks[..., top:ori_h+top, left:ori_w+left]
        pad = (top, left)
    else:
        masks = F.interpolate(masks, original_size, mode="bilinear", align_corners=False)
        pad = None
    return masks, pad

def prompt_and_decoder(args, batched_input, ddp_model, image_embeddings):
    points = (batched_input["point_coords"], batched_input["point_labels"]) \
             if batched_input["point_coords"] is not None else None
    with torch.no_grad():
        sparse_embeddings, dense_embeddings = ddp_model.prompt_encoder(
            points=points,
            boxes=batched_input.get("boxes", None),
            masks=batched_input.get("mask_inputs", None),
        )
        low_res_masks, iou_predictions = ddp_model.mask_decoder(
            image_embeddings=image_embeddings,
            image_pe=ddp_model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=args.multimask,
        )
    if args.multimask:
        max_values, max_indexs = torch.max(iou_predictions, dim=1)
        iou_predictions = max_values.unsqueeze(1)
        low_res_masks = torch.stack([low_res_masks[i:i+1, idx]
                                     for i, idx in enumerate(max_indexs)], 0)
    masks = F.interpolate(low_res_masks, (args.image_size, args.image_size),
                          mode="bilinear", align_corners=False)
    return masks, low_res_masks, iou_predictions

def extract_lcc(binary_map):
    if binary_map.sum() == 0: return binary_map
    mask_u8 = binary_map.astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if n <= 1: return binary_map
    return (labels == (1 + np.argmax(stats[1:, cv2.CC_STAT_AREA]))).astype(np.float32)

def calc_hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or not gt.any(): return 256.0
    pe, ge = pred ^ binary_erosion(pred), gt ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]
    d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return 256.0
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_cbl(pred_bin, gt_bin):
    if gt_bin.sum() == 0: return None
    ys, xs = np.where(gt_bin)
    gt_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + 1e-6
    if pred_bin.sum() == 0: return 0.0
    yp, xp = np.where(pred_bin)
    d = np.sqrt((xp.mean()-xs.mean())**2 + (yp.mean()-ys.mean())**2)
    return float(np.clip(1.0 - d / gt_diag, 0.0, 1.0))

def main(args):
    print('*'*80)
    for k, v in vars(args).items(): print(f"  {k}: {v}")
    print('*'*80)

    model = sam_model_registry[args.model_type](args).to(args.device)
    criterion = FocalDiceloss_IoULoss()
    test_dataset = TestingDataset(
        data_path=args.data_path, image_size=args.image_size,
        mode='test', requires_name=True, point_num=args.point_num,
        return_ori_mask=True, prompt_path=args.prompt_path,
        prompt_mode=args.prompt_mode,
        zoom_ratio=tuple(args.zoom_ratio), shift_ratio=args.shift_ratio)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)
    print(f'Test data: {len(test_loader)}  |  prompt_mode: {args.prompt_mode}')

    model.eval()
    test_loss, test_hd95, test_cbl = [], [], []
    test_iter_metrics = [0] * len(args.metrics)
    l = len(test_loader)

    for batched_input in tqdm(test_loader):
        batched_input = to_device(batched_input, args.device)
        ori_labels   = batched_input["ori_label"]
        original_size = batched_input["original_size"]
        img_name     = batched_input['name'][0]

        with torch.no_grad():
            image_embeddings = model.image_encoder(batched_input["image"])

        batched_input["point_coords"], batched_input["point_labels"] = None, None
        save_path = os.path.join(args.work_dir, args.run_name, f"boxes_{args.prompt_mode}")
        masks, low_res_masks, iou_predictions = prompt_and_decoder(
            args, batched_input, model, image_embeddings)

        masks, pad = postprocess_masks(low_res_masks, args.image_size, original_size)
        if args.save_pred:
            save_masks(masks, save_path, img_name, args.image_size,
                       original_size, pad, batched_input.get("boxes", None), None)

        loss = criterion(masks, ori_labels, iou_predictions)
        test_loss.append(loss.item())

        pred_bin = extract_lcc((masks[0, 0].cpu().numpy() > 0.0).astype(np.float32))
        gt_bin   = ori_labels[0, 0].cpu().numpy().astype(np.float32)
        test_hd95.append(calc_hd95(pred_bin, gt_bin))
        cbl = calc_cbl(pred_bin, gt_bin)
        if cbl is not None: test_cbl.append(cbl)

        bm = SegMetrics(masks, ori_labels, args.metrics)
        for j in range(len(args.metrics)):
            test_iter_metrics[j] += float(bm[j])

    metrics = {args.metrics[i]: f"{test_iter_metrics[i]/l:.4f}" for i in range(len(args.metrics))}
    mean_dice = test_iter_metrics[args.metrics.index('dice')] / l if 'dice' in args.metrics else 0.0
    mean_iou  = test_iter_metrics[args.metrics.index('iou')]  / l if 'iou'  in args.metrics else 0.0
    mean_pre  = test_iter_metrics[args.metrics.index('precision')] / l if 'precision' in args.metrics else 0.0
    mean_rec  = test_iter_metrics[args.metrics.index('recall')]    / l if 'recall'    in args.metrics else 0.0
    mean_hd95 = float(np.mean(test_hd95)) if test_hd95 else 0.0
    mean_cbl  = float(np.mean(test_cbl))  if test_cbl  else 0.0
    print(f"\n[{args.prompt_mode}] loss={np.mean(test_loss):.4f} | "
          f"IoU={mean_iou:.4f} | Dice={mean_dice:.4f} | "
          f"Pre={mean_pre:.4f} | Rec={mean_rec:.4f} | "
          f"HD95={mean_hd95:.2f}px | CBL={mean_cbl:.4f} | N={l}")

    # Luu CSV
    os.makedirs(os.path.join(args.work_dir, "csv"), exist_ok=True)
    csv_path = os.path.join(args.work_dir, "csv", f"sammed2d_{args.prompt_mode}.csv")
    with open(csv_path, "w", newline="") as f_csv:
        writer = csv.writer(f_csv)
        writer.writerow(["model", "prompt_mode", "dice", "iou", "precision", "recall", "hd95", "cbl", "n_samples"])
        writer.writerow(["SAM-Med2D", args.prompt_mode,
                         f"{mean_dice:.4f}", f"{mean_iou:.4f}",
                         f"{mean_pre:.4f}", f"{mean_rec:.4f}",
                         f"{mean_hd95:.4f}", f"{mean_cbl:.4f}", l])
    print(f"CSV saved: {csv_path}")

if __name__ == '__main__':
    args = parse_args()
    main(args)

Overwriting /kaggle/working/SAM-Med2D/test.py


### 2.1 – Kịch bản A: Zoom-Out (~30%)
Bbox được mở rộng đều 4 phía 30% so với tight bbox GT — tương đương Gaussian heatmap "đúng" của PGA.

In [14]:
assert CHECKPOINT, "❌ CHECKPOINT rỗng — chạy cell 12 trước"
!python test.py \
    --work_dir "workdir/test_results" \
    --data_path "dataset_BTXRD" \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --prompt_mode zoom_out \
    --sam_checkpoint {CHECKPOINT} \
    --save_pred True

********************************************************************************
  work_dir: workdir/test_results
  run_name: sammed
  batch_size: 1
  image_size: 256
  device: cuda
  data_path: dataset_BTXRD
  metrics: ['iou', 'dice', 'precision', 'recall']
  model_type: vit_b
  sam_checkpoint: /kaggle/working/drive/MyDrive/model/best_sam.pth
  boxes_prompt: True
  point_num: 1
  iter_point: 1
  multimask: True
  encoder_adapter: True
  prompt_path: None
  save_pred: True
  prompt_mode: zoom_out
  zoom_ratio: [0.15, 0.45]
  shift_ratio: 0.3
********************************************************************************
True
*******load /kaggle/working/drive/MyDrive/model/best_sam.pth
Test data: 232  |  prompt_mode: zoom_out
100%|█████████████████████████████████████████| 232/232 [02:21<00:00,  1.64it/s]

[zoom_out] loss=0.4319 | IoU=0.6130 | Dice=0.7350 | Pre=0.7513 | Rec=0.7478 | HD95=52.92px | CBL=0.8893 | N=232
CSV saved: workdir/test_results/csv/sammed2d_zoom_out.csv


### 2.2 – Kịch bản B: Shift (~30%)
Bbox bị lệch vị trí ~30% so với GT — mô phỏng người dùng chú thích không chính xác.

In [15]:
assert CHECKPOINT, "❌ CHECKPOINT rỗng — chạy cell 12 trước"
!python test.py \
    --work_dir "workdir/test_results" \
    --data_path "dataset_BTXRD" \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --prompt_mode shift \
    --sam_checkpoint {CHECKPOINT} \
    --save_pred True

********************************************************************************
  work_dir: workdir/test_results
  run_name: sammed
  batch_size: 1
  image_size: 256
  device: cuda
  data_path: dataset_BTXRD
  metrics: ['iou', 'dice', 'precision', 'recall']
  model_type: vit_b
  sam_checkpoint: /kaggle/working/drive/MyDrive/model/best_sam.pth
  boxes_prompt: True
  point_num: 1
  iter_point: 1
  multimask: True
  encoder_adapter: True
  prompt_path: None
  save_pred: True
  prompt_mode: shift
  zoom_ratio: [0.15, 0.45]
  shift_ratio: 0.3
********************************************************************************
True
*******load /kaggle/working/drive/MyDrive/model/best_sam.pth
Test data: 232  |  prompt_mode: shift
100%|█████████████████████████████████████████| 232/232 [02:15<00:00,  1.71it/s]

[shift] loss=0.4828 | IoU=0.5818 | Dice=0.7097 | Pre=0.7385 | Rec=0.7117 | HD95=54.61px | CBL=0.8767 | N=232
CSV saved: workdir/test_results/csv/sammed2d_shift.csv


### 2.3 – Kịch bản C: Mixed 70/30
70% zoom-out + 30% shift — kịch bản thực tế người dùng.

In [16]:
assert CHECKPOINT, "❌ CHECKPOINT rỗng — chạy cell 12 trước"
!python test.py \
    --work_dir "workdir/test_results" \
    --data_path "dataset_BTXRD" \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --prompt_mode mixed_7_3 \
    --sam_checkpoint {CHECKPOINT} \
    --save_pred True

********************************************************************************
  work_dir: workdir/test_results
  run_name: sammed
  batch_size: 1
  image_size: 256
  device: cuda
  data_path: dataset_BTXRD
  metrics: ['iou', 'dice', 'precision', 'recall']
  model_type: vit_b
  sam_checkpoint: /kaggle/working/drive/MyDrive/model/best_sam.pth
  boxes_prompt: True
  point_num: 1
  iter_point: 1
  multimask: True
  encoder_adapter: True
  prompt_path: None
  save_pred: True
  prompt_mode: mixed_7_3
  zoom_ratio: [0.15, 0.45]
  shift_ratio: 0.3
********************************************************************************
True
*******load /kaggle/working/drive/MyDrive/model/best_sam.pth
Test data: 232  |  prompt_mode: mixed_7_3
100%|█████████████████████████████████████████| 232/232 [02:15<00:00,  1.71it/s]

[mixed_7_3] loss=0.4454 | IoU=0.6044 | Dice=0.7283 | Pre=0.7491 | Rec=0.7379 | HD95=53.37px | CBL=0.8863 | N=232
CSV saved: workdir/test_results/csv/sammed2d_mixed_7_3.csv


### 2.4 – Tổng hợp kết quả & Visualization
Bảng so sánh 3 kịch bản + hiển thị trực quan trên 3 ảnh đại diện (cùng ảnh với PGA).

In [ ]:
# ── Image-level aggregation từ prediction PNGs ───────────────────────────────
# test.py lưu prediction PNGs per-polygon. Gom theo ảnh gốc → image-level metric.
import cv2, numpy as np, os, csv
from scipy.ndimage import binary_erosion, distance_transform_edt

MODES     = ['zoom_out', 'shift', 'mixed_7_3']
PRED_DIRS = {m: f"workdir/test_results/sammed/boxes_{m}" for m in MODES}
MASK_DIR  = "dataset_BTXRD/test/masks"
S = 256

def calc_m(prob, gt, eps=1e-6):
    pm=(prob>0.5).astype(np.float32); gm=(gt>0.5).astype(np.float32)
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    p,g=pm.astype(bool),gm.astype(bool); hd95=float(S)
    if p.any() and g.any():
        from scipy.ndimage import binary_erosion,distance_transform_edt
        pe=p^binary_erosion(p); ge=g^binary_erosion(g)
        d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
        if len(d1) and len(d2): hd95=float(max(np.percentile(d1,95),np.percentile(d2,95)))
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)),iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),  recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

KEYS = ['dice','iou','precision','recall','hd95','cbl']
print(f'\n{"="*70}')
print(f'  SAM-Med2D Fine-tuned — IMAGE-LEVEL metrics')
print(f'{"="*70}')
for mode in MODES:
    pred_dir = PRED_DIRS[mode]
    if not os.path.exists(pred_dir):
        print(f'  ⚠ {mode}: {pred_dir} không tồn tại — chạy test.py trước'); continue
    stem_data = {}
    for fn in sorted(os.listdir(MASK_DIR)):
        if not fn.endswith('.png'): continue
        parts = fn.rsplit('_',1)
        if len(parts)<2: continue
        stem = parts[0]
        gt_p = cv2.imread(os.path.join(MASK_DIR,fn), cv2.IMREAD_GRAYSCALE)
        pr_p = cv2.imread(os.path.join(pred_dir,fn), cv2.IMREAD_GRAYSCALE)
        if gt_p is None or pr_p is None: continue
        gt256 = cv2.resize(gt_p,(S,S),interpolation=cv2.INTER_NEAREST).astype(np.float32)/255.
        pr256 = cv2.resize(pr_p,(S,S),interpolation=cv2.INTER_NEAREST).astype(np.float32)/255.
        if stem not in stem_data:
            stem_data[stem] = dict(gt=gt256.copy(), prob=pr256.copy())
        else:
            np.maximum(stem_data[stem]['gt'],   gt256, out=stem_data[stem]['gt'])
            np.maximum(stem_data[stem]['prob'], pr256, out=stem_data[stem]['prob'])
    if not stem_data: continue
    recs = [calc_m(d['prob'],d['gt']) for d in stem_data.values()]
    avg  = {k:np.mean([r[k] for r in recs]) for k in KEYS}
    print(f"  [{mode:12s}] N={len(recs)} imgs | Dice={avg['dice']:.4f}  IoU={avg['iou']:.4f}"
          f"  HD95={avg['hd95']:.2f}px  CBL={avg['cbl']:.4f}")
print(f'{"="*70}')
